In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from datasets import load_dataset, Image as HFImage, DatasetDict
import json
import pandas as pd
import matplotlib.pyplot as plt

# Step 1. Load data

In [3]:
dataset = load_dataset("Codatta/MM-Food-100K")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

MM-Food-100K.csv:   0%|          | 0.00/28.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 100000
    })
})


# Step 2. Split the dataset into training/validation/test sets

In [5]:
train_test_split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_val_split = train_test_split['train'].train_test_split(test_size=0.11, seed=42)

# merge back into a single DatasetDict
dataset = DatasetDict({
    'train': train_val_split['train'],
    'validation': train_val_split['test'],
    'test': train_test_split['test']
})

# cast image url's to the real image
dataset = dataset.cast_column('image_url', HFImage())

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 64160
    })
    validation: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 7930
    })
    test: Dataset({
        features: ['image_url', 'camera_or_phone_prob', 'food_prob', 'dish_name', 'food_type', 'ingredients', 'portion_size', 'nutritional_profile', 'cooking_method', 'sub_dt'],
        num_rows: 8010
    })
})


# Step 3. Cast nutrition data to float and add to the dataset

In [6]:
def get_nutrition(example, string):
    """
      Extract calories, proteins, fat, and carbohydrates from the 'nutritional_profile' field
    """
    profile = example['nutritional_profile']
    # Split the string to get the nutrient name (e.g., 'carbohydrate')
    nutrition_name = string.split('_')[0]

    # if it's a string, parse it to a dict
    if isinstance(profile, str):
        try:
            profile = json.loads(profile.replace("'", '"'))
        except:
            return {nutrition_name: None}

    # extract the specific nutritional value using the full string (e.g., 'calories_kcal')
    nutrition_amt = profile.get(string)
    return {nutrition_name: float(nutrition_amt) if nutrition_amt is not None else None}

Extract the nutritional values as separate features

In [7]:
nutrients = ['carbohydrate_g', 'fat_g', 'protein_g', 'calories_kcal']
for n in nutrients:
  # Pass the 'n' variable as an argument to get_nutrition using fn_kwargs
  dataset = dataset.map(get_nutrition, fn_kwargs={'string': n})

Map:   0%|          | 0/64160 [00:00<?, ? examples/s]

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

Map:   0%|          | 0/8010 [00:00<?, ? examples/s]

Map:   0%|          | 0/64160 [00:00<?, ? examples/s]

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

Map:   0%|          | 0/8010 [00:00<?, ? examples/s]

Map:   0%|          | 0/64160 [00:00<?, ? examples/s]

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

Map:   0%|          | 0/8010 [00:00<?, ? examples/s]

In [8]:
# print dataset's features
print(dataset['train'].features)
print(dataset['validation'].features)
print(dataset['test'].features)

{'image_url': Image(mode=None, decode=True), 'camera_or_phone_prob': Value('float64'), 'food_prob': Value('float64'), 'dish_name': Value('string'), 'food_type': Value('string'), 'ingredients': Value('string'), 'portion_size': Value('string'), 'nutritional_profile': Value('string'), 'cooking_method': Value('string'), 'sub_dt': Value('int64'), 'carbohydrate': Value('float64'), 'fat': Value('float64'), 'protein': Value('float64'), 'calories': Value('float64')}
{'image_url': Image(mode=None, decode=True), 'camera_or_phone_prob': Value('float64'), 'food_prob': Value('float64'), 'dish_name': Value('string'), 'food_type': Value('string'), 'ingredients': Value('string'), 'portion_size': Value('string'), 'nutritional_profile': Value('string'), 'cooking_method': Value('string'), 'sub_dt': Value('int64'), 'carbohydrate': Value('float64'), 'fat': Value('float64'), 'protein': Value('float64'), 'calories': Value('float64')}
{'image_url': Image(mode=None, decode=True), 'camera_or_phone_prob': Value('

Get the maximum values for each nutritional profile to normalize them

In [9]:
cal_stats = pd.Series(dataset['train']['calories']).describe().round(2)
prot_stats = pd.Series(dataset['train']['protein']).describe().round(2)
fat_stats = pd.Series(dataset['train']['fat']).describe().round(2)
carb_stats = pd.Series(dataset['train']['carbohydrate']).describe().round(2)

MAX_CALORIES = cal_stats['max']
MAX_PROTEINS = prot_stats['max']
MAX_FAT = fat_stats['max']
MAX_CARBS = carb_stats['max']

# Step 4. Process images and create DataLoaders

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
import torch

# define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


def custom_transform(examples):
    # process images
    examples['pixel_values'] = [transform(img.convert("RGB")) for img in examples['image_url']]

    # normalize all nutritional values
    examples['labels_cal'] = [float(cal) / MAX_CALORIES for cal in examples['calories']]
    examples['labels_prot'] = [float(prot) / MAX_PROTEINS for prot in examples['protein']]
    examples['labels_fat'] = [float(fat) / MAX_FAT for fat in examples['fat']]
    examples['labels_carb'] = [float(carb) / MAX_CARBS for carb in examples['carbohydrate']]

    return examples

# apply the transform
dataset.set_transform(custom_transform)

# Hugging Face -> PyTorch DataLoader
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.stack([
            torch.tensor([x['labels_cal'], x['labels_prot'], x['labels_fat'], x['labels_carb']])
            for x in batch
        ])
    }

# create the DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(dataset['train'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
val_loader = DataLoader(dataset['validation'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset['test'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Step 5. Load weights of the pre-trained ResNet152 on Food2K

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
# initialize the standard architecture
model = models.resnet152(weights=None)

# change the 'fc' layer to 2000 to satisfy the .pth file (pre-trained model on Food2K dataset)
in_features = model.fc.in_features #2048
model.fc = nn.Linear(in_features, 2000) # change from 2048->1000 to 2048->2000

# load the weights from pre-trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_dict = torch.load('/content/drive/MyDrive/MIE1517 Project/resnet152_shared/food2k_resnet152_0.0001.pth', map_location=device)

# clean 'module.' prefix if it exists from DataParallel training
new_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

model.load_state_dict(new_state_dict)
model.to(device)

# strip the last layer to create feature extractor
modules = list(model.children())[:-1]
resnet152_features = nn.Sequential(*modules)

resnet152_features.to(device)
resnet152_features.eval()

# Step 6. Feature Extraction
Note: this process takes a very long time as it passes all 100,000 images through the ResNet and saves them to Google Drive. It took over 15 hours on the provided GPU on colab.

In [ ]:
import os
import time
from PIL import Image, ImageFile
import itertools

save_dir_features = "/content/drive/MyDrive/MIE1517 Project/resnet152_features"
os.makedirs(save_dir_features, exist_ok=True)

def save_resnet152_features(dataloader, loadername):
    print(f'Starting feature extraction for: {loadername}...')

    # Determine the starting batch index by checking existing files
    existing_files = [f for f in os.listdir(save_dir_features) if f.startswith(f'{loadername}_') and f.endswith('.pt')]
    saved_indices = []
    for f in existing_files:
        try:
            # Extract index from filename like 'train_0.pt', 'train_1.pt', etc.
            idx_str = f.replace(f'{loadername}_', '').replace('.pt', '')
            saved_indices.append(int(idx_str))
        except ValueError:
            print(f"Warning: Could not parse index from filename {f}. Skipping.")
            continue

    start_index = 0
    if saved_indices:
        start_index = max(saved_indices) + 1
        print(f"Found {len(saved_indices)} existing files for {loadername}. Resuming from batch {start_index}.")
    else:
        print(f"No existing feature files found for {loadername}. Starting from batch 0.")

    with torch.no_grad():
        # Create an iterator for the dataloader and skip already processed batches
        dataloader_iter = itertools.islice(dataloader, start_index, None)

        # Iterate through the remaining batches
        for i_relative, batch in enumerate(dataloader_iter):
            i_absolute = start_index + i_relative # Calculate the absolute batch index

            file_path = os.path.join(save_dir_features, f'{loadername}_{i_absolute}.pt')

            # Double-check: if for some reason file exists (e.g., race condition or manual file addition)
            if os.path.exists(file_path):
                print(f"Warning: Batch {i_absolute} for {loadername} unexpectedly exists (after calculating start_index). Skipping.")
                continue

            img = batch['pixel_values'].to(device)
            label = batch['labels'].to(device)

            features = resnet152_features(img)
            features = features.view(features.size(0), -1)

            torch.save((features.cpu(), label.cpu()), file_path)

            if i_absolute % 2 == 0:
                print(f"Batch {i_absolute} for {loadername} saved...")

save_resnet152_features(train_loader, 'train')
save_resnet152_features(val_loader, 'val')
save_resnet152_features(test_loader, 'test')

Starting feature extraction for: train...
Found 21 existing files for train. Resuming from batch 21.
Batch 22 for train saved...
Batch 24 for train saved...
Batch 26 for train saved...


KeyboardInterrupt: 